## Importaciones

In [ ]:
from funciones_BBDD import *
import pandas as pd

# DRUGBANK 1
DataFrame general

In [ ]:
xml_file = "./drugbank_all_full_database.xml/full_database.xml"
output_file = "enzimas.csv"

csv_enzimas(output_file, xml_file)

In [ ]:
df_drugbank = pd.read_csv ('enzimas.csv')

print("Principios activos:", len(df_drugbank["drugbank_id"].unique()))
df_drugbank.head()

### Descripcion
1. **drugbank_id** Identificador único de DrugBank para el principio activo
2. **drug_name** Nombre del principio activo (genérico)
3. **synonimous** Nombres alternativos para el principio activo (comerciales/ codigos de investigacion/ en otros idiomas)
4. **type** Categoria de la proteína. Rol biologico de la proteína con la que interactúa el principio activo
    - Target. Proteina diana, donde el farmaco ejerce su efecto principal
    - Enzyme. proteinas que metabolizan el fármaco
5. **Gene_name** Símbolo del gen que codifica esa proteína
6. **uniprot_id** Codigo de acceso para UniProt
7. **Action** Accion del principio activo a la proteína
  


In [ ]:
df_drugbank['type'].unique()

In [ ]:
df_drugbank['action'].unique()

In [ ]:
df_drugbank.isna().any()

In [ ]:
#No hay elementos de accion nulos solo con las enzimas
enz = df_drugbank[df_drugbank["type"]=="enzyme"]
enz.isna().any()

# DRUGBANK 2
DF con los codigos ATC (se añadió mas tarde cuando se vio que era importante para establecer la prioridad)

In [ ]:
xml_file = "./drugbank_all_full_database.xml/full_database.xml"
output_file = "ATC_DB.csv"

csv_ATC_db(xml_file, output_file)

In [ ]:
df_ATC = pd.read_csv ('ATC_DB.csv')

print("Principios activos:", len(df_ATC["drugbank_id"].unique()))
df_ATC.head()

### Descripcion
1. **drugbank_id**
2. **drug_name**
3. **atc_codes**

# SIDDER (efectos adversos)

In [ ]:
path_se="meddra_all_se.tsv",
path_freq="meddra_freq.tsv",
path_names="drug_names.tsv",
output_csv="sidder_cambio.csv"

#Filtrados por prefered term (PT)
csv_sidder(path_se, path_freq, path_names, output_csv)

In [ ]:
df_sidder = pd.read_csv ('sidder_cambio.csv')

print("Diferentes id:", len(df_drugbank["flat"].unique()))
df_drugbank.head()

### Descripcion
1. **flat**
2. **drug_name**
3. **side_effect_se**
4. **freq_mean_y**

In [ ]:
df_drugbank.isna().any()

# CIMA

In [ ]:
#Con este comando obetenemos: medicamentos.csv
main()

In [ ]:
#Abrimos el csv de CIMA
med = pd.read_csv("medicamentos.csv")
med.head()

### Descripcion
1. **nombre_medicamento**
2. **numero_resgistro**
3. **principios_activos** (pueden tener mas de uno)
4. **codigo_atc** (solo hay uno)
5. **forma_farmaceutica**
6. **situacion_registro**

In [ ]:
med.isna().any()

# Establecemos prioridad de enzimas
Para ello se usa la base de datos de CIMA

In [ ]:
# Elimino nulos que solo hay en principios activos enrealidad
med = med.dropna(subset=["principios_activos"])
# Me quedo solo con las que tengan un principio activo
uno = med[~med["principios_activos"].str.contains("/", na=False)]
#Me quedo con un solo medicamento  para ver la prioridad de enzima de metabolizacion que tiene un principio activo
prioridad = uno.drop_duplicates(subset=["principios_activos"], keep = 'first')

print(f"Los medicamentos existentes en CIMA son: {len(med)}")
print(f"Los medicamentos con tan solo un principio activo resgistrados en CIMA son: {len(uno)}")
print(f"Solo un medicamento con un principio activo: {len(prioridad)}")

In [ ]:
# Lo intentamos con el nombre del principio activo: no funciono
# Lo intenamos con los sinonimos tambien: tampoco se pudo
# Se optó por los codigos ATC para identificar los principios activos entre bases de datos diferentes

In [ ]:
# Diccionario que cuenta cuantas encimas tengo de cada y cuales son el top 5 con las que me quedare
diccionario_enzimas = (enzimas[enzimas["type"] == "enzyme"]["gene_name"].value_counts().to_dict())

print(diccionario_enzimas)

In [ ]:
#Por tanto escogemos las que se metabolizan por cualquiera de las siguientes enzimas ["CYP2D6", "CYP3A4", "CYP3A5", "CYP2C19", "CYP2C9", "CYP1A2"] 
#que son las top 5 y CYP3A5 porque va con CYP3A4 (explicar mejor)

#Me quedo solo con las enzimas
enzimas = df_drugbank[df_drugbank["type"]=="enzyme"]
#Las que queremos (pongo tambien 3A5)
top_5 = enzimas[enzimas["gene_name"].isin(["CYP2D6", "CYP3A4", "CYP3A5", "CYP2C19", "CYP2C9", "CYP1A2"])]
#Las que se metabolizan por mas de uno manteniendo la primera que son de las que queremos ver la prioridad
duplicados = top_5[top_5.duplicated(subset="drugbank_id", keep=False)]
#Dejamos un unico id para que quede constancia de cuantos son
final = duplicados["drug_name"].unique().tolist()

print(f"Los principios activos que se metabolizan por mas de una enzima de las tenidas en cuenta: {len(final)}")

# De esas 755 solo me quedo con los que tienen como accion sustrato. Para ello primero separo las acciones porque pueden tener mas de una
print(f"Las acciones que hay en los principios activos que se metabolizan por mas de una enzima son: {duplicados['action'].unique()}")
# Separamos los codigos ATC de lo que nos importa para los df (los duplicados)
acciones_sep_enz = duplicados.copy()
acciones_sep_enz["action"] = acciones_sep_enz["action"].str.split(r"\|")
acciones_sep_enz = acciones_sep_enz.explode("action")
# Nos quedamos solo con los sustratos
sustratos = acciones_sep_enz[acciones_sep_enz["action"]== "substrate"]
print(f"Las enzimas de las que vamos a intentar obtener prioridad son: {len(sustratos['drug_name'].unique())}")

In [ ]:
#Con el df que obtuvimos luego de los codigos ATC de drugbank
#Elimino los que no tengan código ATC
codigos_ATC = df_ATC.dropna(subset=['atc_codes'])
codigos_ATC.tail()

### Separo los códigos ATC
Porque puede haber mas de uno por principio activo (como hicimos con las acciones del primer df otenido de drugbank)

In [ ]:
#Los separo
separado = codigos_ATC["atc_codes"].str.split(r"\|")
#Donde almaceno las filas
rows = []
#identificador de la fila donde me llego
i = 0
#Por cada codigo
for cods in separado:
    fila = codigos_ATC.iloc[i]
    if len(cods)!=1:
        d_id = fila["drugbank_id"]
        name = fila["drug_name"]
        fda = fila["fda_app_n"]
        for uno in cods:
            rows.append([d_id, name, fda, uno])
    else:
        rows.append(fila.tolist())
    i+=1

ATC_separado = pd.DataFrame(rows, columns=[
        "drugbank_id",
        "drug_name",
        "fda_app_n",
        "atc_codes"
    ])

ATC_separado.head()

In [ ]:
#Llamamos db a los que coincidan que tengan ATC de los sustratos en DrugBank
db = pd.merge(sustratos, ATC_separado, on= "drug_name", how="inner")
print(f"Los principios activos con código ATC de los que queremos obtener prioridad son: {len(db['drug_name'].unique())}")
#Llamamos total a los principios activos que tienen ATC y coincide que hay el mismo ATC en CIMA (no hay mas de 1 ATC registrado en la misma celda en CIMA)
total = pd.merge(db, CIMA, left_on="atc_codes", right_on="codigo_atc", how="inner")
print(f"De:{len(sustratos['drug_name'].unique())} principios de los que queremos obtener prioridad solo podemos intentarlo finalmente con:{len(total['drugbank_id_x'].unique())}")

In [ ]:
#Me quedo solo con un numero de registro pero luego si es necesario reviso los totales
df_unico = total.drop_duplicates(subset="drug_name")

# PDFs
Se obtienen solo los pdfs de los principios activos que contienen ATC en drugbank y que coincide con los ATC obtenidos de CIMA.  
Se divide en df resultante en 11 csv con 4 medicamentos cada uno, excepto el 11 que solo tiene 18.  
Se ejecutan en varias celdas de código debido a que se ejecutaron en dias diferentes para evitar los errores 429 (too many requests).  
Los archivos no encontrados (error 404) no se pudieron incluir y se detalla debajo de cada celda de código.  
Se buscan de forma programatica gracias al número de registro

In [ ]:
#Separacion de dfs
bloques = dividir_df(df_unico, 40)
for idx, bloque in enumerate(bloques):
    bloque.to_csv(f"bloque_{idx+1}.csv", index=False)

In [ ]:
#Se pudieron encontrar todos
pdfs("medicamentos_1","bloque_1.csv")

In [ ]:
#Se encontraron 38
pdfs("medicamentos_2","bloque_2.csv")

In [ ]:
#TODOS
pdfs("medicamentos_3","bloque_3.csv")

In [ ]:
#TODOS
pdfs("medicamentos_4","bloque_4.csv")

In [ ]:
#39
pdfs("medicamentos_5","bloque_5.csv")

In [ ]:
#TODOS
pdfs("medicamentos_6","bloque_6.csv")

In [ ]:
#TODOS 
pdfs("medicamentos_7","bloque_7.csv")

In [ ]:
#38
pdfs("medicamentos_8","bloque_8.csv")

In [ ]:
#36
pdfs("medicamentos_9","bloque_9.csv")

In [ ]:
#39
pdfs("medicamentos_10","bloque_10.csv")

In [ ]:
#TODOS
pdfs("medicamentos_11","bloque_11.csv")

# NOTEBOOKLM Y CHATGPT
Se utiliza notebookLM para introducir los pdfs de las fichas tecnicas obtenidas con CIMA y ChatGPT para la generación de los df siguientes con las informacion proporcionada en NotebookLM

In [ ]:
import pandas as pd

data = [
    ["Acetaminophen", None, 0],

    ["Amitriptyline", "CYP2D6", 1],
    ["Amitriptyline", "CYP2C19", 1],

    ["Argatroban", None, 0],

    ["Atomoxetine", "CYP2D6", 1],

    ["Betaxolol", None, 0],

    ["Bortezomib", "CYP3A4", 1],
    ["Bortezomib", "CYP2C19", 1],
    ["Bortezomib", "CYP1A2", 1],

    ["Bupivacaine", None, 0],

    ["Caffeine", "CYP1A2", 1],

    ["Citalopram", None, 0],

    ["Clotrimazole", None, 0],

    ["Codeine", "CYP2D6", 1],

    ["Conjugated estrogens", None, 0],

    ["Cyclosporine", "CYP3A4", 1],

    ["Desogestrel", "CYP3A", 1],

    ["Disopyramide", "CYP3A", 1],

    ["Eletriptan", "CYP3A4", 1],

    ["Erythromycin", "CYP3A4", 1],

    ["Fluorometholone", None, 0],

    ["Fluvoxamine", "CYP2D6", 1],

    ["Gefitinib", "CYP3A4", 1],
    ["Gefitinib", "CYP2D6", 1],

    ["Indinavir", "CYP3A4", 1],

    ["Indomethacin", None, 0],

    ["Lidocaine", None, 0],

    ["Lovastatin", "CYP3A4", 1],

    ["Methadone", "CYP3A4", 1],

    ["Metoprolol", "CYP2D6", 1],

    ["Nevirapine", "CYP3A4", 1],

    ["Nicotine", None, 0],

    ["Oxiconazole", None, 0],

    ["Pantoprazole", "CYP2C19", 1],

    ["Phenytoin", "CYP2C9", 1],
    ["Phenytoin", "CYP2C19", 1],

    ["Ranolazine", "CYP3A4", 1],

    ["Reboxetine", "CYP3A4", 1],

    ["Ropivacaine", "CYP1A2", 1],

    ["Sildenafil", "CYP3A4", 1],

    ["Theophylline", "CYP1A2", 1],

    ["Ticlopidine", None, 0],

    ["Tramadol", "CYP2D6", 1],

    ["Valproic acid", "UGT1A6", 1],
    ["Valproic acid", "UGT1A9", 1],
    ["Valproic acid", "UGT2B7", 1],

    ["Venlafaxine", "CYP2D6", 1],
]

med_1 = pd.DataFrame(data, columns=["farmaco", "enzima", "prioridad"])

print(len(med_1["farmaco"].unique()))

In [ ]:
import pandas as pd

data = [
    ["Albendazole", None, 0],
    ["Alprazolam", None, 0],
    ["Amlodipine", None, 0],
    ["Beclomethasone dipropionate", None, 0],
    ["Betamethasone", None, 0],

    ["Celecoxib", "CYP2C9", 1],

    ["Chlorpromazine", None, 0],

    ["Clobazam", "CYP3A4", 1],

    ["Clozapine", "CYP1A2", 1],

    ["Dextromethorphan", "CYP2D6", 1],

    ["Diltiazem", "CYP3A4", 1],

    ["Duloxetine", None, 0],

    ["Fluoxetine", "CYP2D6", 1],

    ["Flutamide", "CYP1A2", 1],

    ["Haloperidol", None, 0],

    ["Imipramine", "CYP3A4", 1],
    ["Imipramine", "CYP2C19", 1],
    ["Imipramine", "CYP1A2", 1],

    ["Lansoprazole", "CYP2C19", 1],

    ["Lercanidipine", "CYP3A4", 1],

    ["Levonorgestrel", None, 0],

    ["Loratadine", "CYP3A4", 1],
    ["Loratadine", "CYP2D6", 1],

    ["Meperidine", None, 0],

    ["Mexiletine", None, 0],

    ["Mirtazapine", None, 0],

    ["Montelukast", None, 0],

    ["Nabumetone", None, 0],

    ["Olanzapine", "CYP1A2", 1],

    ["Omeprazole", "CYP2C19", 1],

    ["Oxycodone", "CYP3A4", 1],

    ["Palonosetron", None, 0],

    ["Progesterone", None, 0],

    ["Ritonavir", "CYP3A", 1],

    ["Sorafenib", "CYP3A4", 1],

    ["Sulfadiazine", None, 0],

    ["Timolol", None, 0],

    ["Trimethoprim", None, 0],

    ["Vinorelbine", "CYP3A4", 1],

    ["Zidovudine", None, 0],

    ["Zolpidem", "CYP3A4", 1],
]

med_2 = pd.DataFrame(data, columns=["farmaco", "enzima", "prioridad"])

print(len(med_2["farmaco"].unique()))

In [ ]:
import pandas as pd

data = [
    ["Aprepitant", "CYP3A4", 1],

    ["Bosentan", "CYP3A4", 1],
    ["Bosentan", "CYP2C9", 1],

    ["Carbamazepine", "CYP3A4", 1],

    ["Chloroquine", None, 0],

    ["Cinnarizine", "CYP2D6", 1],

    ["Clonidine", None, 0],

    ["Cyclophosphamide", None, 0],

    ["Daunorubicin", "Aldo-ceto reductasas citoplasmáticas", 1],

    ["Diclofenac", None, 0],

    ["Doxazosin", "CYP3A4", 1],

    ["Efavirenz", "CYP3A4", 1],
    ["Efavirenz", "CYP2B6", 1],

    ["Eplerenone", "CYP3A4", 1],

    ["Erlotinib", "CYP3A4", 1],

    ["Ethosuximide", "CYP3A4", 1],

    ["Fenfluramine", "CYP1A2", 1],
    ["Fenfluramine", "CYP2B6", 1],
    ["Fenfluramine", "CYP2D6", 1],

    ["Fluorouracil", "Dihidropirimidina deshidrogenasa (DPD)", 1],

    ["Galantamine", "CYP2D6", 1],
    ["Galantamine", "CYP3A4", 1],

    ["Hydroxyzine", "Alcohol deshidrogenasa", 1],
    ["Hydroxyzine", "CYP3A4", 1],

    ["Imatinib", "CYP3A4", 1],

    ["Ivermectin", "CYP3A4", 1],

    ["Labetalol", "Glucuronidación (conjugación)", 1],

    ["Losartan", "CYP2C9", 1],

    ["Medroxyprogesterone acetate", "CYP3A4", 1],

    ["Midazolam", "CYP3A4", 1],

    ["Nicardipine", None, 0],

    ["Nortriptyline", "CYP1A2", 1],
    ["Nortriptyline", "CYP2C", 1],
    ["Nortriptyline", "CYP2D6", 1],
    ["Nortriptyline", "CYP3A4", 1],

    ["Prednisone", "Glucuronidación", 1],
    ["Prednisone", "Sulfatación", 1],

    ["Propranolol", None, 0],

    ["Rifabutin", None, 0],

    ["Simvastatin", "CYP3A4", 1],

    ["Tamoxifen", "CYP3A4", 1],

    ["Testosterone", "5α-reductasa", 1],

    ["Thiopental", None, 0],

    ["Trazodone", "CYP3A4", 1],

    ["Triamcinolone", None, 0],

    ["Verapamil", "CYP3A4", 1],
    ["Verapamil", "CYP1A2", 1],
    ["Verapamil", "CYP2C8", 1],
    ["Verapamil", "CYP2C9", 1],
    ["Verapamil", "CYP2C18", 1],

    ["Vinblastine", "CYP3A4", 1],

    ["Vincristine", "CYP3A4", 1],

    ["Voriconazole", "CYP2C19", 1],
    ["Voriconazole", "CYP2C9", 1],
    ["Voriconazole", "CYP3A4", 1],

    ["Warfarin", None, 0],
]

med_3 = pd.DataFrame(data, columns=["farmaco", "enzima", "prioridad"])

print(len(med_3["farmaco"].unique()))

In [ ]:
import pandas as pd

data = [
    ["Alfentanil", "CYP3A4", 1],

    ["Apomorphine", None, 0],

    ["Bimatoprost", None, 0],

    ["Clopidogrel", "CYP2C19", 1],

    ["Diazepam", "CYP3A4", 1],
    ["Diazepam", "CYP2C19", 1],

    ["Disulfiram", None, 0],

    ["Donepezil", "CYP3A4", 1],

    ["Epinastine", None, 0],

    ["Esomeprazole", "CYP2C19", 1],

    ["Estradiol", "CYP1A2", 1],
    ["Estradiol", "CYP3A4", 1],
    ["Estradiol", "CYP1B1", 1],
    ["Estradiol", "CYP2C9", 1],

    ["Etoposide", "CYP3A4", 1],

    ["Hydrocortisone", None, 0],

    ["Irinotecan", "CYP3A", 1],

    ["Loperamide", "CYP3A4", 1],
    ["Loperamide", "CYP2C8", 1],

    ["Mefenamic acid", "CYP2C9", 1],

    ["Meloxicam", "CYP2C9", 1],

    ["Metronidazole", None, 0],

    ["Mifepristone", "CYP3A4", 1],

    ["Modafinil", None, 0],

    ["Mometasone", "CYP3A4", 1],

    ["Naproxen", None, 0],

    ["Nateglinide", "CYP2C9", 1],

    ["Norethisterone", None, 0],

    ["Ondansetron", "CYP3A4", 1],
    ["Ondansetron", "CYP2D6", 1],
    ["Ondansetron", "CYP1A2", 1],

    ["Paroxetine", None, 0],

    ["Pentamidine", None, 0],

    ["Perphenazine", None, 0],

    ["Primidone", None, 0],

    ["Propofol", None, 0],

    ["Ranitidine", None, 0],

    ["Risperidone", "CYP2D6", 1],

    ["Sirolimus", "CYP3A4", 1],

    ["Tacrolimus", "CYP3A4", 1],

    ["Tamsulosin", "CYP3A4", 1],
    ["Tamsulosin", "CYP2D6", 1],

    ["Terbinafine", None, 0],

    ["Tretinoin", "CYP26A1", 1],
    ["Tretinoin", "CYP3A4", 1],

    ["Triazolam", None, 0],

    ["Trimipramine", None, 0],

    ["Vardenafil", "CYP3A4", 1],

    ["Zonisamide", None, 0],
]

med_4 = pd.DataFrame(data, columns=["farmaco", "enzima", "prioridad"])

print(len(med_4["farmaco"].unique()))

In [ ]:
import pandas as pd

data = [
    ["Acetylsalicylic acid", None, 0],

    ["Almotriptan", "MAO-A", 1],

    ["Atazanavir", "CYP3A4", 1],

    ["Atorvastatin", "CYP3A4", 1],

    ["Azelastine", None, 0],

    ["Benzocaine", "Colinesterasas", 1],

    ["Buprenorphine", "CYP3A4", 1],

    ["Cinacalcet", "CYP3A4", 1],
    ["Cinacalcet", "CYP1A2", 1],

    ["Cyclobenzaprine", None, 0],

    ["Diphenhydramine", None, 0],

    ["Doxorubicin", None, 0],

    ["Ethinylestradiol", None, 0],

    ["Felodipine", "CYP3A4", 1],

    ["Fluocinonide", None, 0],

    ["Fluvastatin", None, 0],

    ["Formoterol", None, 0],

    ["Glyburide", None, 0],

    ["Glycopyrronium", "CYP2D6", 1],

    ["Guanfacine", None, 0],

    ["Ibuprofen", None, 0],

    ["Irbesartan", "CYP2C9", 1],

    ["Ketoconazole", None, 0],

    ["Leflunomide", None, 0],

    ["Letrozole", "CYP2A6", 1],
    ["Letrozole", "CYP3A4", 1],

    ["Levobupivacaine", "CYP3A4", 1],
    ["Levobupivacaine", "CYP1A2", 1],

    ["Maprotiline", "CYP2D6", 1],

    ["Melatonin", "CYP1A", 1],

    ["Methylprednisolone", "CYP3A4", 1],

    ["Mycophenolic acid", "UGT1A9", 1],

    ["Nitrendipine", None, 0],

    ["Norgestimate", None, 0],

    ["Oxybutynin", None, 0],

    ["Pimozide", "CYP3A4", 1],
    ["Pimozide", "CYP2D6", 1],

    ["Salmeterol", None, 0],

    ["Selegiline", "CYP2B6", 1],

    ["Sulfamethoxazole", None, 0],

    ["Thalidomide", None, 0],

    ["Tipranavir", "CYP3A4", 1],

    ["Tolterodine", "CYP2D6", 1],
]

med_5 = pd.DataFrame(data, columns=["farmaco", "enzima", "prioridad"])

print(len(med_5["farmaco"].unique()))

In [ ]:
import pandas as pd

data = [
    ["Amiodarone", "CYP3A4", 1],

    ["Anastrozole", None, 0],

    ["Aripiprazole", "CYP2D6", 1],
    ["Aripiprazole", "CYP3A4", 1],

    ["Atovaquone", None, 0],

    ["Bicalutamide", None, 0],

    ["Brinzolamide", "CYP3A4", 1],

    ["Budesonide", "CYP3A4", 1],

    ["Bupropion", "CYP2B6", 1],

    ["Carvedilol", "CYP2D6", 1],
    ["Carvedilol", "CYP2C9", 1],

    ["Cilostazol", "CYP3A4", 1],

    ["Clarithromycin", None, 0],

    ["Clindamycin", None, 0],

    ["Clomipramine", "CYP3A4", 1],
    ["Clomipramine", "CYP2C19", 1],
    ["Clomipramine", "CYP1A2", 1],

    ["Dexamethasone", None, 0],

    ["Docetaxel", None, 0],

    ["Domperidone", "CYP3A4", 1],

    ["Doxepin", None, 0],

    ["Escitalopram", "CYP2C19", 1],

    ["Finasteride", None, 0],

    ["Flecainide", "CYP2D6", 1],

    ["Gemfibrozil", None, 0],

    ["Gliclazide", None, 0],

    ["Idarubicin", None, 0],

    ["Ifosfamide", "CYP3A4", 1],

    ["Itraconazole", None, 0],

    ["Ketamine", "CYP3A4", 1],

    ["Metoclopramide", None, 0],

    ["Moclobemide", "CYP2C19", 1],
    ["Moclobemide", "CYP2D6", 1],

    ["Naloxone", None, 0],

    ["Nifedipine", "CYP3A4", 1],

    ["Paclitaxel", "CYP2C8", 1],
    ["Paclitaxel", "CYP3A4", 1],

    ["Phenobarbital", None, 0],

    ["Proguanil", "CYP2C19", 1],

    ["Propafenone", "CYP2D6", 1],

    ["Quetiapine", "CYP3A4", 1],

    ["Rabeprazole", "CYP2C19", 1],
    ["Rabeprazole", "CYP3A4", 1],

    ["Rifaximin", "CYP3A4", 1],

    ["Saquinavir", "CYP3A4", 1],

    ["Sertraline", None, 0],

    ["Zopiclone", "CYP3A4", 1],
]

med_6 = pd.DataFrame(data, columns=["farmaco", "enzima", "prioridad"])

print(len(med_6["farmaco"].unique()))

In [ ]:
import pandas as pd

data = [
    ["Abiraterone", "CYP3A4", 1],

    ["Acenocoumarol", "CYP2C9", 1],

    ["Alogliptin", None, 0],

    ["Apremilast", "CYP3A4", 1],

    ["Brivaracetam", "Amidasa hepática y extra-hepática", 1],

    ["Bromazepam", None, 0],

    ["Cariprazine", "CYP3A4", 1],

    ["Cenobamate", None, 0],

    ["Ciclesonide", "CYP3A4", 1],

    ["Clevidipine", "Esterasas", 1],

    ["Cobimetinib", "CYP3A4", 1],
    ["Cobimetinib", "UGT2B7", 1],

    ["Dapoxetine", "CYP2D6", 1],
    ["Dapoxetine", "CYP3A4", 1],
    ["Dapoxetine", "FMO1", 1],

    ["Darunavir", "CYP3A4", 1],

    ["Dasatinib", "CYP3A4", 1],

    ["Dihydrocodeine", "CYP2D6", 1],

    ["Dronedarone", "CYP3A4", 1],

    ["Etoricoxib", "CYP3A4", 1],

    ["Everolimus", "CYP3A4", 1],

    ["Febuxostat", "CYP", 1],
    ["Febuxostat", "UDPGT", 1],

    ["Flunarizine", "CYP2D6", 1],

    ["Fusidic acid", None, 0],

    ["Hydroxychloroquine", None, 0],

    ["Lapatinib", "CYP3A4", 1],

    ["Lopinavir", "CYP3A", 1],

    ["Mianserin", None, 0],

    ["Nebivolol", "CYP2D6", 1],

    ["Nilotinib", "CYP3A4", 1],

    ["Ospemifene", None, 0],

    ["Paliperidone", None, 0],

    ["Pirfenidone", None, 0],

    ["Roflumilast", "CYP3A4", 1],
    ["Roflumilast", "CYP1A2", 1],

    ["Rotigotine", None, 0],

    ["Sertindole", "CYP2D6", 1],
    ["Sertindole", "CYP3A", 1],

    ["Solifenacin", "CYP3A4", 1],

    ["Sunitinib", "CYP3A4", 1],

    ["Tiotropium", None, 0],

    ["Tirbanibulin", None, 0],

    ["Trabectedin", None, 0],

    ["Trastuzumab emtansine", None, 0],

    ["Zuclopenthixol", None, 0],
]

med_7 = pd.DataFrame(data, columns=["farmaco", "enzima", "prioridad"])

print(len(med_7["farmaco"].unique()))

In [ ]:
import pandas as pd

data = [
    ["Agomelatine", "CYP1A2", 1],

    ["Ambrisentan", "CYP3A4", 1],

    ["Apixaban", "CYP3A4", 1],

    ["Asenapine", "UGT1A4", 1],
    ["Asenapine", "CYP1A2", 1],

    ["Avanafil", None, 0],

    ["Axitinib", "CYP3A4", 1],

    ["Cabazitaxel", "CYP3A", 1],

    ["Cabozantinib", "CYP3A4", 1],

    ["Capsaicin", None, 0],

    ["Clomethiazole", None, 0],

    ["Crizotinib", "CYP3A4", 1],

    ["Dapagliflozin", "UGT1A9", 1],

    ["Desvenlafaxine", "CYP3A4", 1],

    ["Etravirine", None, 0],

    ["Fesoterodine", None, 0],

    ["Gestodene", "CYP3A4", 1],

    ["Ivacaftor", "CYP3A4", 1],

    ["Lacosamide", None, 0],

    ["Maribavir", None, 0],

    ["Midostaurin", None, 0],

    ["Mirabegron", None, 0],

    ["Nalmefene", None, 0],

    ["Parecoxib", None, 0],

    ["Pazopanib", "CYP3A4", 1],

    ["Perampanel", "CYP3A", 1],

    ["Prasugrel", None, 0],

    ["Rilpivirine", "CYP3A", 1],

    ["Rivaroxaban", None, 0],

    ["Ruxolitinib", None, 0],

    ["Safinamide", None, 0],

    ["Saxagliptin", "CYP3A4", 1],

    ["Tapentadol", "UGT1A6", 1],
    ["Tapentadol", "UGT1A9", 1],
    ["Tapentadol", "UGT2B7", 1],

    ["Temsirolimus", None, 0],

    ["Ticagrelor", "CYP3A4", 1],

    ["Tofacitinib", "CYP3A4", 1],

    ["Ulipristal", None, 0],

    ["Vemurafenib", "CYP3A4", 1],

    ["Vismodegib", None, 0],
]

med_8 = pd.DataFrame(data, columns=["farmaco", "enzima", "prioridad"])

print(len(med_8["farmaco"].unique()))

In [ ]:
import pandas as pd

data = [
    ["Acalabrutinib", "CYP3A4", 1],

    ["Brexpiprazole", None, 0],

    ["Cannabidiol", "CYP2C19", 1],
    ["Cannabidiol", "CYP3A4", 1],

    ["Ceritinib", "CYP3A4", 1],

    ["Cobicistat", None, 0],

    ["Dabrafenib", "CYP3A4", 1],

    ["Dasabuvir", "CYP2C8", 1],

    ["Dexibuprofen", None, 0],

    ["Difluocortolone", None, 0],

    ["Elbasvir", None, 0],

    ["Eliglustat", None, 0],

    ["Enzalutamide", "CYP2C8", 1],

    ["Fluticasone furoate", "CYP3A4", 1],

    ["Fruquintinib", "CYP3A4", 1],

    ["Ibrutinib", "CYP3A4", 1],

    ["Idelalisib", "Aldehído oxidasa", 1],

    ["Isavuconazole", "CYP3A4", 1],

    ["Lenvatinib", "Aldehído oxidasa", 1],
    ["Lenvatinib", "CYP3A4", 1],

    ["Macitentan", "CYP3A4", 1],

    ["Manidipine", None, 0],

    ["Methylene blue", None, 0],

    ["Naloxegol", None, 0],

    ["Olaparib", "CYP3A4", 1],
    ["Olaparib", "CYP3A5", 1],

    ["Opium", None, 0],

    ["Osimertinib", "CYP3A4", 1],
    ["Osimertinib", "CYP3A5", 1],

    ["Paritaprevir", "CYP3A4", 1],

    ["Pitolisant", None, 0],

    ["Pomalidomide", None, 0],

    ["Ponatinib", "Esterasas o amidasas", 1],

    ["Regorafenib", "CYP3A4", 1],
    ["Regorafenib", "UGT1A9", 1],

    ["Rupatadine", "CYP3A4", 1],

    ["Selumetinib", "CYP3A4", 1],

    ["Stiripentol", "CYP1A2", 1],
    ["Stiripentol", "CYP2C19", 1],
    ["Stiripentol", "CYP3A4", 1],

    ["Tegafur", "CYP2A6", 1],

    ["Treosulfan", None, 0],

    ["Vortioxetine", None, 0],
]

med_9 = pd.DataFrame(data, columns=["farmaco", "enzima", "prioridad"])

print(len(med_9["farmaco"].unique()))

In [ ]:
import pandas as pd

data = [
    ["Abemaciclib", "CYP3A4", 1],

    ["Alpelisib", "CYP3A4", 1],

    ["Apalutamide", "CYP2C8", 1],
    ["Apalutamide", "CYP3A4", 1],

    ["Asciminib", "CYP3A4", 1],
    ["Asciminib", "UGT2B7", 1],
    ["Asciminib", "UGT2B17", 1],

    ["Avatrombopag", None, 0],

    ["Binimetinib", "UGT1A1", 1],

    ["Brigatinib", None, 0],

    ["Clobetasol", None, 0],

    ["Dacomitinib", "CYP2D6", 1],

    ["Deflazacort", "estearasas plasmáticas", 1],

    ["Doravirine", None, 0],

    ["Ebastine", "CYP3A4", 1],

    ["Encorafenib", "CYP3A4", 1],

    ["Erdafitinib", "CYP2C9", 1],
    ["Erdafitinib", "CYP3A4", 1],

    ["Esketamine", "CYP2B6", 1],
    ["Esketamine", "CYP3A4", 1],

    ["Etrasimod", "CYP2C8", 1],
    ["Etrasimod", "CYP2C9", 1],
    ["Etrasimod", "CYP3A4", 1],

    ["Fedratinib", "CYP3A4", 1],

    ["Fluticasone", "CYP3A4", 1],

    ["Glecaprevir", None, 0],

    ["Hydrocortisone aceponate", None, 0],

    ["Hydrocortisone probutate", None, 0],

    ["Ivosidenib", "CYP3A4", 1],

    ["Letermovir", None, 0],

    ["Lorlatinib", "CYP3A4", 1],
    ["Lorlatinib", "UGT1A4", 1],

    ["Momelotinib", "CYP3A4", 1],

    ["Mometasone furoate", None, 0],

    ["Osilodrostat", None, 0],

    ["Polatuzumab vedotin", None, 0],

    ["Ponesimod", None, 0],

    ["Quizartinib", "CYP3A4", 1],

    ["Remdesivir", "carboxilesterasa 1", 1],
    ["Remdesivir", "catepsina A", 1],

    ["Rimegepant", None, 0],

    ["Rucaparib", None, 0],

    ["Seladelpar", None, 0],

    ["Siponimod", "CYP2C9", 1],

    ["Sparsentan", None, 0],

    ["Tezacaftor", None, 0],

    ["Trifarotene", "CYP2C9", 1],
    ["Trifarotene", "CYP3A4", 1],
    ["Trifarotene", "CYP2C8", 1],

    ["Voxilaprevir", "CYP3A4", 1],
]

med_10 = pd.DataFrame(data, columns=["farmaco", "enzima", "prioridad"])

print(len(med_10["farmaco"].unique()))

In [ ]:
import pandas as pd

data = [
    ["Abrocitinib", "CYP2C19", 1],
    ["Abrocitinib", "CYP2C9", 1],

    ["Avacopan", None, 0],

    ["Avapritinib", None, 0],

    ["Deucravacitinib", "CYP1A2", 1],
    ["Deucravacitinib", "CES2 (carboxilesterasa 2)", 1],
    ["Deucravacitinib", "UGT (uridina glucuroniltransferasa)", 1],
    ["Deucravacitinib", "CYP2B6/2D6", 1],

    ["Elexacaftor", "CYP3A4", 1],

    ["Fezolinetant", "CYP1A2", 1],

    ["Futibatinib", "CYP3A4", 1],

    ["Linzagolix", None, 0],

    ["Mavacamten", "CYP2C19", 1],

    ["Pirtobrutinib", "CYP3A4", 1],
    ["Pirtobrutinib", "UGT1A8", 1],
    ["Pirtobrutinib", "UGT1A9", 1],

    ["Ripretinib", "CYP3A4/5", 1],

    ["Risdiplam", None, 0],

    ["Ritlecitinib", "Glutatión S-transferasa (GST)", 1],
    ["Ritlecitinib", "CYP3A4", 1],
    ["Ritlecitinib", "CYP2C8", 1],
    ["Ritlecitinib", "CYP1A2", 1],
    ["Ritlecitinib", "CYP2C9", 1],

    ["Selpercatinib", "CYP3A4", 1],

    ["Sotorasib", "CYP2C8", 1],
    ["Sotorasib", "CYP3A4", 1],

    ["Upadacitinib", "CYP3A4", 1],

    ["Vamorolone", None, 0],

    ["Zanubrutinib", None, 0],
]

med_11 = pd.DataFrame(data, columns=["farmaco", "enzima", "prioridad"])

print(len(med_11["farmaco"].unique()))

In [ ]:
#Unimos los 11 bloques
med_total = pd.concat([med_1,med_2,med_3,med_4,med_5,med_6,med_7,med_8,med_9,med_10,med_11])
# Elimino nulos y los que no se encuentran en los top 5 declarados
med_total = med_total.dropna()
med_total = med_total[med_total["enzima"].isin(["CYP2D6", "CYP3A4", "CYP3A5", "CYP2C19", "CYP2C9", "CYP1A2"])]
med_total
# le cambio los nombres y los guardo en: excepciones.csv
med_total.columns = ["Drug_name","Gene_name","Prioridad"]
med_total.to_csv("excepciones.csv", index=False)